In [ ]:
```xml
<VSCode.Cell language="markdown">
# 💥 Chaos Engineering & Resilience Testing Guide

**Proactively test system reliability through controlled failure injection and chaos experiments**

This guide covers:
- Chaos engineering principles
- Failure injection strategies
- Chaos experiment design
- Observability for chaos testing
- Recovery validation
- Blast radius control
- Automated chaos scenarios
- Lessons from chaos experiments
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Chaos Engineering Fundamentals

### Chaos Experiment Framework
```python
from dataclasses import dataclass
from typing import Dict, List, Optional, Callable
from enum import Enum
from datetime import datetime, timedelta
import random

class FailureType(Enum):
    """Types of failures to inject"""
    LATENCY = "latency"  # Slow response
    ERROR = "error"  # Service errors
    PACKET_LOSS = "packet_loss"  # Network packet loss
    DISK_FULL = "disk_full"  # Storage failure
    CPU_SPIKE = "cpu_spike"  # Resource exhaustion
    MEMORY_SPIKE = "memory_spike"  # OOM conditions
    SERVICE_DOWN = "service_down"  # Complete outage
    DEGRADATION = "degradation"  # Partial degradation

@dataclass
class ChaosExperiment:
    """Define a chaos experiment"""
    name: str
    description: str
    target_service: str
    failure_type: FailureType
    failure_rate: float  # 0.0-1.0: % of requests affected
    duration_seconds: int
    delay_ms: int = 0
    error_code: int = 500
    blast_radius: str = "canary"  # canary, regional, global
    rollback_enabled: bool = True
    max_duration_seconds: int = 300
    created_at: datetime = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now()
    
    def is_within_blast_radius(self, deployment_size: int) -> bool:
        """Check if blast radius is acceptable"""
        if self.blast_radius == "canary":
            return deployment_size <= 10  # Max 10 instances
        elif self.blast_radius == "regional":
            return deployment_size <= 50
        else:  # global
            return True
    
    def should_auto_rollback(self, error_rate: float) -> bool:
        """Determine if experiment should auto-rollback"""
        if not self.rollback_enabled:
            return False
        
        # Rollback if error rate > 2x expected
        return error_rate > (self.failure_rate * 2)

class ChaosExperimentResult:
    """Result of chaos experiment"""
    
    def __init__(self, experiment: ChaosExperiment):
        self.experiment = experiment
        self.started_at: Optional[datetime] = None
        self.ended_at: Optional[datetime] = None
        self.requests_total: int = 0
        self.requests_failed: int = 0
        self.requests_success: int = 0
        self.latency_p50_ms: float = 0
        self.latency_p99_ms: float = 0
        self.max_latency_ms: float = 0
        self.errors_observed: Dict[int, int] = {}  # error_code -> count
        self.insights: List[str] = []
        self.rollback_triggered: bool = False
    
    def calculate_error_rate(self) -> float:
        """Calculate observed error rate"""
        if self.requests_total == 0:
            return 0
        return self.requests_failed / self.requests_total
    
    def add_insight(self, insight: str):
        """Add finding from experiment"""
        self.insights.append(insight)
    
    def generate_report(self) -> Dict:
        """Generate experiment report"""
        return {
            'experiment': self.experiment.name,
            'duration_seconds': (self.ended_at - self.started_at).total_seconds() if self.ended_at else 0,
            'requests_total': self.requests_total,
            'error_rate': f"{self.calculate_error_rate():.1%}",
            'latency_p50': f"{self.latency_p50_ms}ms",
            'latency_p99': f"{self.latency_p99_ms}ms",
            'max_latency': f"{self.max_latency_ms}ms",
            'rollback_triggered': self.rollback_triggered,
            'insights': self.insights
        }

class ChaosOrchestrator:
    """Orchestrate chaos experiments"""
    
    def __init__(self):
        self.experiments: List[ChaosExperiment] = []
        self.results: List[ChaosExperimentResult] = []
        self.active_experiment: Optional[ChaosExperiment] = None
    
    def schedule_experiment(self, experiment: ChaosExperiment):
        """Add experiment to schedule"""
        self.experiments.append(experiment)
    
    def run_experiment(self, experiment: ChaosExperiment) -> ChaosExperimentResult:
        """Execute chaos experiment"""
        result = ChaosExperimentResult(experiment)
        result.started_at = datetime.now()
        
        self.active_experiment = experiment
        
        print(f"🔥 Starting chaos experiment: {experiment.name}")
        print(f"   Target: {experiment.target_service}")
        print(f"   Failure type: {experiment.failure_type.value}")
        print(f"   Failure rate: {experiment.failure_rate:.1%}")
        print(f"   Duration: {experiment.duration_seconds}s")
        
        # Simulate experiment
        for _ in range(experiment.duration_seconds):
            # Simulate requests
            requests_this_second = random.randint(10, 100)
            
            for _ in range(requests_this_second):
                result.requests_total += 1
                
                # Randomly trigger failures based on failure_rate
                if random.random() < experiment.failure_rate:
                    result.requests_failed += 1
                    
                    if experiment.failure_type == FailureType.ERROR:
                        result.errors_observed[experiment.error_code] = \
                            result.errors_observed.get(experiment.error_code, 0) + 1
                    
                    if experiment.failure_type == FailureType.LATENCY:
                        latency = experiment.delay_ms + random.randint(0, 500)
                    else:
                        latency = random.randint(10, 100)
                else:
                    result.requests_success += 1
                    latency = random.randint(5, 50)
                
                # Track latencies
                if _ == 0:
                    result.latency_p50_ms = latency
                result.max_latency_ms = max(result.max_latency_ms, latency)
        
        # Check if should rollback
        if experiment.should_auto_rollback(result.calculate_error_rate()):
            result.rollback_triggered = True
            result.add_insight(f"Auto-rollback triggered (error rate: {result.calculate_error_rate():.1%})")
        
        result.ended_at = datetime.now()
        self.results.append(result)
        self.active_experiment = None
        
        return result

# Usage
chaos = ChaosOrchestrator()

experiment1 = ChaosExperiment(
    name="API Latency Injection",
    description="Inject 500ms latency on 10% of requests",
    target_service="user-api",
    failure_type=FailureType.LATENCY,
    failure_rate=0.1,
    duration_seconds=60,
    delay_ms=500,
    blast_radius="canary"
)

result = chaos.run_experiment(experiment1)
print(result.generate_report())
```

### Failure Injection Strategies
```python
class FailureInjector:
    """Inject failures into system"""
    
    def __init__(self, service_name: str):
        self.service_name = service_name
        self.injected_failures: List[Dict] = []
    
    def inject_latency(self, endpoint: str, latency_ms: int, percentage: float):
        """Add latency to endpoint"""
        self.injected_failures.append({
            'type': 'latency',
            'endpoint': endpoint,
            'latency_ms': latency_ms,
            'percentage': percentage,
            'active': True
        })
    
    def inject_errors(self, endpoint: str, error_code: int, percentage: float):
        """Return errors from endpoint"""
        self.injected_failures.append({
            'type': 'error',
            'endpoint': endpoint,
            'error_code': error_code,
            'percentage': percentage,
            'active': True
        })
    
    def simulate_network_partition(self, target_service: str, duration_seconds: int):
        """Simulate network split"""
        self.injected_failures.append({
            'type': 'network_partition',
            'target': target_service,
            'duration_seconds': duration_seconds,
            'active': True
        })
    
    def degrade_service(self, capability: str, performance_factor: float):
        """Degrade service capability (e.g., 0.5 = 50% slower)"""
        self.injected_failures.append({
            'type': 'degradation',
            'capability': capability,
            'performance_factor': performance_factor,
            'active': True
        })
    
    def get_active_failures(self) -> List[Dict]:
        """Get currently active failures"""
        return [f for f in self.injected_failures if f['active']]
    
    def clear_failures(self):
        """Stop all injected failures"""
        for failure in self.injected_failures:
            failure['active'] = False
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Observability & Recovery Validation

### Metrics Collection During Chaos
```python
class ChaosMetricsCollector:
    """Collect metrics during chaos experiments"""
    
    def __init__(self):
        self.metrics: Dict[str, List[float]] = {}
        self.service_health: Dict[str, Dict] = {}
    
    def collect_response_time(self, service: str, latency_ms: float):
        """Record response time"""
        if service not in self.metrics:
            self.metrics[service] = []
        
        self.metrics[service].append(latency_ms)
    
    def collect_error_rate(self, service: str, errors: int, total: int):
        """Record error rate"""
        error_rate = errors / total if total > 0 else 0
        
        if f"{service}_error_rate" not in self.metrics:
            self.metrics[f"{service}_error_rate"] = []
        
        self.metrics[f"{service}_error_rate"].append(error_rate)
    
    def collect_dependency_health(self, service: str, dependency: str, healthy: bool):
        """Record dependency health"""
        if service not in self.service_health:
            self.service_health[service] = {}
        
        self.service_health[service][dependency] = 'healthy' if healthy else 'unhealthy'
    
    def get_percentile(self, service: str, percentile: int) -> Optional[float]:
        """Get percentile latency"""
        if service not in self.metrics:
            return None
        
        sorted_values = sorted(self.metrics[service])
        index = int(len(sorted_values) * percentile / 100)
        
        return sorted_values[index] if sorted_values else None
    
    def analyze_recovery(self, service: str, expected_recovery_time_ms: int = 5000) -> bool:
        """Check if service recovered within expected time"""
        # Simplified: check if error rate returned to near-zero
        if f"{service}_error_rate" not in self.metrics:
            return False
        
        recent_errors = self.metrics[f"{service}_error_rate"][-10:]
        avg_recent_error_rate = sum(recent_errors) / len(recent_errors) if recent_errors else 0
        
        return avg_recent_error_rate < 0.01  # Less than 1% error rate

class ResilienceValidator:
    """Validate system resilience properties"""
    
    def __init__(self):
        self.validations: List[Dict] = []
    
    def validate_timeout_behavior(self, service: str, expected_timeout_ms: int, observed_max_ms: int) -> bool:
        """Validate timeout is enforced"""
        is_valid = observed_max_ms <= expected_timeout_ms * 1.1  # Allow 10% margin
        
        self.validations.append({
            'check': 'timeout_enforcement',
            'service': service,
            'passed': is_valid,
            'expected': expected_timeout_ms,
            'observed': observed_max_ms
        })
        
        return is_valid
    
    def validate_circuit_breaker(self, service: str, failure_rate: float, expected_to_trip: bool) -> bool:
        """Validate circuit breaker trips at threshold"""
        # Typically trips at 50% error rate
        threshold = 0.5
        should_trip = failure_rate > threshold
        is_valid = should_trip == expected_to_trip
        
        self.validations.append({
            'check': 'circuit_breaker',
            'service': service,
            'passed': is_valid,
            'failure_rate': failure_rate,
            'should_trip': should_trip
        })
        
        return is_valid
    
    def validate_graceful_degradation(self, service: str, primary_failed: bool, fallback_available: bool) -> bool:
        """Validate fallback is used when primary fails"""
        is_valid = (not primary_failed) or fallback_available
        
        self.validations.append({
            'check': 'graceful_degradation',
            'service': service,
            'passed': is_valid,
            'primary_failed': primary_failed,
            'fallback_available': fallback_available
        })
        
        return is_valid
    
    def get_resilience_score(self) -> float:
        """Calculate overall resilience score"""
        if not self.validations:
            return 0.0
        
        passed = sum(1 for v in self.validations if v['passed'])
        return passed / len(self.validations)
```

### Automated Runbooks
```python
class AutomatedRunbook:
    """Automated recovery procedures"""
    
    def __init__(self, name: str):
        self.name = name
        self.steps: List[Dict] = []
        self.triggered_count: int = 0
        self.success_count: int = 0
    
    def add_step(self, action: str, condition: Optional[Callable] = None, retry_count: int = 1):
        """Add recovery step"""
        self.steps.append({
            'action': action,
            'condition': condition,
            'retry_count': retry_count,
            'completed': False
        })
    
    def execute(self) -> bool:
        """Execute recovery runbook"""
        self.triggered_count += 1
        
        for step in self.steps:
            # Check condition if provided
            if step['condition'] and not step['condition']():
                continue
            
            # Execute step with retries
            for attempt in range(step['retry_count']):
                print(f"⚙️  Executing: {step['action']} (attempt {attempt + 1})")
                
                try:
                    # In real implementation, call actual recovery action
                    step['completed'] = True
                    break
                except Exception as e:
                    print(f"   ❌ Failed: {e}")
                    if attempt < step['retry_count'] - 1:
                        print(f"   🔄 Retrying...")
        
        success = all(step['completed'] for step in self.steps)
        
        if success:
            self.success_count += 1
            print(f"✅ Runbook {self.name} completed successfully")
        else:
            print(f"❌ Runbook {self.name} failed")
        
        return success
    
    def get_success_rate(self) -> float:
        """Get runbook success rate"""
        if self.triggered_count == 0:
            return 0.0
        return self.success_count / self.triggered_count
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Chaos Experiments Library

### Common Scenarios
```python
class ChaosScenarios:
    """Pre-built chaos test scenarios"""
    
    @staticmethod
    def database_connection_pool_exhaustion() -> ChaosExperiment:
        """Test behavior when DB connections exhausted"""
        return ChaosExperiment(
            name="Database Connection Pool Exhaustion",
            description="Simulate all DB connections in use",
            target_service="api-server",
            failure_type=FailureType.ERROR,
            failure_rate=0.3,
            duration_seconds=120,
            error_code=503,
            blast_radius="canary"
        )
    
    @staticmethod
    def cascading_failure() -> List[ChaosExperiment]:
        """Test cascade protection (Bulkhead pattern)"""
        return [
            ChaosExperiment(
                name="Cascade Test Phase 1 - Degrade Service A",
                description="Degrade upstream service to test isolation",
                target_service="service-a",
                failure_type=FailureType.DEGRADATION,
                failure_rate=0.5,
                duration_seconds=60,
                blast_radius="canary"
            ),
            ChaosExperiment(
                name="Cascade Test Phase 2 - Verify Service B Isolation",
                description="Verify dependent service remains healthy",
                target_service="service-b",
                failure_type=FailureType.ERROR,
                failure_rate=0.0,  # Should not see errors if isolated
                duration_seconds=60,
                blast_radius="canary"
            )
        ]
    
    @staticmethod
    def network_partition() -> ChaosExperiment:
        """Test behavior during network split"""
        return ChaosExperiment(
            name="Network Partition",
            description="Simulate split-brain network partition",
            target_service="cluster-node",
            failure_type=FailureType.SERVICE_DOWN,
            failure_rate=1.0,
            duration_seconds=30,
            blast_radius="canary",
            rollback_enabled=True
        )
    
    @staticmethod
    def resource_exhaustion() -> ChaosExperiment:
        """Test under resource pressure"""
        return ChaosExperiment(
            name="Memory Spike - OOM Simulation",
            description="Simulate out-of-memory condition",
            target_service="cache-layer",
            failure_type=FailureType.MEMORY_SPIKE,
            failure_rate=0.0,  # All requests affected
            duration_seconds=60,
            blast_radius="canary"
        )
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Chaos Engineering Checklist

✅ **Experiment Design**
- [ ] Hypothesis defined
- [ ] Blast radius limited
- [ ] Rollback plan ready
- [ ] Success criteria clear
- [ ] Metrics identified

✅ **Execution Safety**
- [ ] Canary deployment used
- [ ] Auto-rollback enabled
- [ ] Monitoring active
- [ ] On-call team aware
- [ ] Runbook prepared

✅ **Observability**
- [ ] Metrics collected
- [ ] Logs available
- [ ] Traces enabled
- [ ] Alerts configured
- [ ] Dashboards visible

✅ **Results & Learning**
- [ ] Results documented
- [ ] Insights recorded
- [ ] Weaknesses identified
- [ ] Improvements prioritized
- [ ] Lessons shared

✅ **Continuous Chaos**
- [ ] Experiments automated
- [ ] Scheduled regularly
- [ ] Results tracked
- [ ] Regression detected
- [ ] Resilience improving
</VSCode.Cell>
```